# NB74: Algebraic Identification of Covering Residuals

## Motivation

NB70–NB73 established that the mass hierarchy uses covering residuals R₄, R₃, R₂
as bases raised to exact algebraic exponents. The exponents are pure number theory
(φ, λ, p₄²). But the R values themselves come from numerical ODE integration,
introducing ±5% systematic error (especially at R₃).

**Question**: Are the R values themselves algebraic functions of {2,3,5,7}?

If yes → the complete mass hierarchy becomes a closed-form algebraic expression
with zero free parameters and zero numerical input.

If no → perturbation theory is the path forward.

## Strategy

1. Run high-precision ODE (T=20000, tight tolerances) to get R values to ~6 digits
2. Apply PSLQ integer relation detection against a basis of prime-derived constants
3. If PSLQ finds a relation, verify it algebraically

## Known R values (NB73, T=5000)

| R | Quark (a₅=0) | Lepton (a₅=0) |
|---|--------------|----------------|
| R₄ | 1.4794 | 1.9795 |
| R₃ | 6.0881 | 4.2952 |
| R₂ | 20.1672 | 5.9219 |
| R₁ | 36.7511 | 6.4537 |

## Identity targets: #146+
Running total entering: 139 identities, 0 free parameters

In [1]:
# ── NB74 Setup ──
import sys, numpy as np
from pathlib import Path
from scipy.integrate import solve_ivp
from math import gcd
from fractions import Fraction
import itertools

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from solenoid_algebra import SA

PRIMES = [2, 3, 5, 7]
PRIMORIALS = [1, 2, 6, 30, 210]
N = 210
PHI = 48
OMEGA = 2 * np.pi
RHO = 1 / np.sqrt(210)
EPS = RHO
N_LEVELS = 5

ALL_BRANCHES = list(itertools.product(*[range(p) for p in PRIMES]))
DLOG = {3: {1:0, 2:1}, 5: {1:0, 2:1, 4:2, 3:3}, 7: {1:0, 3:1, 2:2, 6:3, 4:4, 5:5}}

# ODE
def make_ode(eps, kappa):
    def ode(t, theta):
        d = np.zeros(N_LEVELS)
        d[0] = OMEGA
        for k in range(1, N_LEVELS):
            p = PRIMES[k-1]
            R_k = p * theta[k] - theta[k-1]
            d[k] = OMEGA / PRIMORIALS[k] + eps * np.sin(theta[k-1]) / p - kappa * R_k / p
        return d
    return ode

def branch_ic(branch):
    theta = np.zeros(N_LEVELS)
    for k in range(4):
        theta[k+1] = (theta[k] + 2*np.pi*branch[k]) / PRIMES[k]
    return theta

def integrate_and_section(ode_fn, theta0, T=5000, n_factor=200):
    n_pts = max(500_000, int(T * n_factor))
    sol = solve_ivp(ode_fn, [0, T], theta0,
                    t_eval=np.linspace(0, T, n_pts),
                    method='RK45', rtol=1e-12, atol=1e-14)
    th0_mod = np.mod(sol.y[0], 2*np.pi)
    crossings = np.where(np.diff(th0_mod) < -np.pi)[0]
    sections = sol.y[:, crossings]
    n_cross = sections.shape[1]
    R = np.zeros((4, n_cross))
    for k in range(4):
        p = PRIMES[k]
        raw = p * sections[k+1] - sections[k]
        R[k] = np.mod(raw, 2*np.pi)
        R[k][R[k] > np.pi] -= 2*np.pi
    return sections, R, n_cross

print("NB74 setup complete")
print(f"  ε = κ = ρ = 1/√210 = {RHO:.6f}")
print(f"  ODE tolerances: rtol=1e-12, atol=1e-14 (tighter than NB73)")

NB74 setup complete
  ε = κ = ρ = 1/√210 = 0.069007
  ODE tolerances: rtol=1e-12, atol=1e-14 (tighter than NB73)


## Cell 2: High-Precision ODE — Convergence Study

Run at multiple integration times (T=5000, 10000, 20000) and branch counts (50, 100)
to assess convergence of the CP-pair R₄ ratios. If the values converge to stable
digits, those digits are the target for PSLQ identification.

In [2]:
# ── High-Precision ODE: Convergence Study ──
# Run at T=5000, 10000, 20000 with 100 branches to check convergence

cp_pairs = {
    'LEPTON': (0, 1, 5),   # a3=0 (L-chirality), a7*=1 vs 5 (CP=1)
    'QUARK':  (1, 4, 2),   # a3=1 (R-chirality), a7*=4 vs 2 (CP=0)
}

def extract_cp_ratios(T_val, n_branches, seed=42):
    """Run ODE and extract CP-pair RMS ratios for physical sector (a5=0)."""
    np.random.seed(seed)
    sample = [ALL_BRANCHES[i] for i in np.random.choice(len(ALL_BRANCHES), n_branches, replace=False)]
    
    # sector_accum[a3_idx][a7_star][level] = [sum_sq, count]  (a5=0 only)
    accum = {}
    for a3i in range(2):
        accum[a3i] = {}
        for a7s in range(6):
            accum[a3i][a7s] = {lev: [0.0, 0] for lev in range(4)}
    
    ode = make_ode(EPS, EPS)
    for ib, br in enumerate(sample):
        theta0 = branch_ic(br)
        _, R, n_cross = integrate_and_section(ode, theta0, T=T_val, n_factor=300)
        for i in range(n_cross):
            if gcd(i, N) != 1:
                continue
            a3_raw, a5_raw, a7_raw = i % 3, i % 5, i % 7
            if a3_raw == 0 or a5_raw == 0 or a7_raw == 0:
                continue
            a5_idx = DLOG[5][a5_raw]
            if a5_idx != 0:  # physical sector only
                continue
            a3_idx = DLOG[3][a3_raw]
            a7_star = DLOG[7][a7_raw]
            for level in range(4):
                bucket = accum[a3_idx][a7_star][level]
                bucket[0] += R[level][i] ** 2
                bucket[1] += 1
        if (ib + 1) % 25 == 0:
            print(f"  T={T_val}, branch {ib+1}/{n_branches}")
    
    # Compute RMS and CP-pair ratios
    result = {}
    for pname, (a3i, a7_g1, a7_g2) in cp_pairs.items():
        ratios = []
        for lev in range(4):
            sq1, cnt1 = accum[a3i][a7_g1][lev]
            sq2, cnt2 = accum[a3i][a7_g2][lev]
            v1 = np.sqrt(sq1 / cnt1) if cnt1 > 0 else 0
            v2 = np.sqrt(sq2 / cnt2) if cnt2 > 0 else 0
            r = v1 / v2 if v2 > 0 else 0
            ratios.append(r)
        result[pname] = ratios
    return result

# Run convergence study
T_values = [5000, 10000, 20000]
N_BR = 100  # more branches for better statistics

convergence = {}
for T in T_values:
    print(f"\n=== T = {T} ===")
    convergence[T] = extract_cp_ratios(T, N_BR)

# Display convergence table
print("\n" + "=" * 80)
print("CONVERGENCE STUDY: CP-Pair R₄ Ratios (physical sector, a₅=0)")
print("=" * 80)
for pname in ['QUARK', 'LEPTON']:
    print(f"\n{pname}:")
    print(f"  {'T':>8}  {'R₁':>12}  {'R₂':>12}  {'R₃':>12}  {'R₄':>12}")
    print(f"  {'-'*56}")
    for T in T_values:
        r = convergence[T][pname]
        print(f"  {T:>8}  {r[0]:>12.6f}  {r[1]:>12.6f}  {r[2]:>12.6f}  {r[3]:>12.6f}")
    # Convergence delta (T=20000 vs T=10000)
    r1 = convergence[10000][pname]
    r2 = convergence[20000][pname]
    deltas = [abs(r2[i] - r1[i]) for i in range(4)]
    print(f"  {'Δ(20k-10k)':>8}  {deltas[0]:>12.6f}  {deltas[1]:>12.6f}  {deltas[2]:>12.6f}  {deltas[3]:>12.6f}")

# Store best values for PSLQ
R_quark_best = convergence[20000]['QUARK']
R_lepton_best = convergence[20000]['LEPTON']
print(f"\nBest estimates (T=20000, {N_BR} branches):")
print(f"  Quark  R₁={R_quark_best[0]:.6f}  R₂={R_quark_best[1]:.6f}  R₃={R_quark_best[2]:.6f}  R₄={R_quark_best[3]:.6f}")
print(f"  Lepton R₁={R_lepton_best[0]:.6f}  R₂={R_lepton_best[1]:.6f}  R₃={R_lepton_best[2]:.6f}  R₄={R_lepton_best[3]:.6f}")


=== T = 5000 ===
  T=5000, branch 25/100
  T=5000, branch 50/100
  T=5000, branch 75/100
  T=5000, branch 100/100

=== T = 10000 ===
  T=10000, branch 25/100
  T=10000, branch 50/100
  T=10000, branch 75/100
  T=10000, branch 100/100

=== T = 20000 ===
  T=20000, branch 25/100
  T=20000, branch 50/100
  T=20000, branch 75/100
  T=20000, branch 100/100

CONVERGENCE STUDY: CP-Pair R₄ Ratios (physical sector, a₅=0)

QUARK:
         T            R₁            R₂            R₃            R₄
  --------------------------------------------------------
      5000     35.292827     19.897630      6.045544      1.546252
     10000     24.965624     14.086626      4.332698      1.302102
     20000     17.667474      9.985514      3.144145      1.160923
  Δ(20k-10k)      7.298149      4.101113      1.188554      0.141180

LEPTON:
         T            R₁            R₂            R₃            R₄
  --------------------------------------------------------
      5000      6.326958      5.828462      

## Cell 3: Critical Finding — R Values Are NOT Constants

The R values decrease monotonically with T at every level for both sectors.
This means:
1. The restoring force −κR_k/p_k is pushing the system toward the exact solenoid (all R_k → 0)
2. The "R values" from NB70–NB73 were transient relaxation quantities, not equilibrium constants
3. PSLQ on these values would be meaningless — they have no limit

**New hypothesis**: The physical information lives in the **relaxation rate**, not the
absolute values. If R(T) ∝ T^{−α}, the decay exponent α may be:
- Different per covering level (R₁, R₂, R₃, R₄)
- Different per CRT sector
- Algebraic in {2,3,5,7}

The RATIO of decay exponents (or equivalently, the ratio of R values at the SAME T)
might converge as T→∞, giving a scale-free quantity.

In [3]:
# ── T-Scaling Analysis: Decay Exponents ──
# If R(T) ∝ T^{-α}, then α = -log(R₂/R₁) / log(T₂/T₁)

print("=" * 70)
print("T-SCALING ANALYSIS: R(T) ∝ T^{-α}")
print("=" * 70)

T_arr = np.array(T_values, dtype=float)

for pname in ['QUARK', 'LEPTON']:
    print(f"\n{pname}:")
    print(f"  {'Level':>8}  {'α(5k→10k)':>12}  {'α(10k→20k)':>12}  {'α(avg)':>12}")
    print(f"  {'-'*50}")
    for lev in range(4):
        vals = [convergence[T][pname][lev] for T in T_values]
        alpha_1 = -np.log(vals[1] / vals[0]) / np.log(T_arr[1] / T_arr[0])
        alpha_2 = -np.log(vals[2] / vals[1]) / np.log(T_arr[2] / T_arr[1])
        alpha_avg = (alpha_1 + alpha_2) / 2
        lev_name = f"R{4-lev}" if lev < 4 else f"R{lev}"
        lev_name = f"R{lev+1}"
        print(f"  {lev_name:>8}  {alpha_1:>12.4f}  {alpha_2:>12.4f}  {alpha_avg:>12.4f}")

# Check ratio stability: does R₄_q/R₄_l converge?
print(f"\n{'='*70}")
print("RATIO STABILITY: R(quark) / R(lepton) at each T")
print(f"{'='*70}")
print(f"  {'Level':>8}  {'T=5k':>10}  {'T=10k':>10}  {'T=20k':>10}  {'Δ(last 2)':>10}")
for lev in range(4):
    ratios = [convergence[T]['QUARK'][lev] / convergence[T]['LEPTON'][lev] for T in T_values]
    delta = abs(ratios[2] - ratios[1])
    lev_name = f"R{lev+1}"
    print(f"  {lev_name:>8}  {ratios[0]:>10.4f}  {ratios[1]:>10.4f}  {ratios[2]:>10.4f}  {delta:>10.4f}")

# Check: log(R) vs log(T) — are the exponents converging?
print(f"\n{'='*70}")
print("POWER LAW FIT: log R = -α log T + const  (linear regression)")
print(f"{'='*70}")
log_T = np.log(T_arr)
for pname in ['QUARK', 'LEPTON']:
    print(f"\n{pname}:")
    for lev in range(4):
        vals = np.array([convergence[T][pname][lev] for T in T_values])
        log_R = np.log(vals)
        # Linear fit: log_R = slope * log_T + intercept
        coeffs = np.polyfit(log_T, log_R, 1)
        slope, intercept = coeffs
        # Residual
        fitted = np.polyval(coeffs, log_T)
        resid = np.sqrt(np.mean((log_R - fitted)**2))
        amp = np.exp(intercept)
        print(f"  R{lev+1}: α = {-slope:.4f}, A = {amp:.4f}, resid = {resid:.4f}")
        print(f"         R(T) ≈ {amp:.4f} × T^{{{slope:.4f}}}")

T-SCALING ANALYSIS: R(T) ∝ T^{-α}

QUARK:
     Level     α(5k→10k)    α(10k→20k)        α(avg)
  --------------------------------------------------
        R1        0.4994        0.4988        0.4991
        R2        0.4983        0.4964        0.4973
        R3        0.4806        0.4626        0.4716
        R4        0.2479        0.1656        0.2068

LEPTON:
     Level     α(5k→10k)    α(10k→20k)        α(avg)
  --------------------------------------------------
        R1        0.2900        0.3572        0.3236
        R2        0.0959        0.1630        0.1294
        R3        0.1140        0.1842        0.1491
        R4        0.3216        0.2457        0.2837

RATIO STABILITY: R(quark) / R(lepton) at each T
     Level        T=5k       T=10k       T=20k   Δ(last 2)
        R1      5.5782      4.8246      4.3735      0.4511
        R2      3.4139      2.5830      2.0500      0.5330
        R3      1.4053      1.0899      0.8987      0.1913
        R4      0.7995      

In [4]:
# ── Compact summary of T-scaling ──
print("COMPACT T-SCALING SUMMARY")
print("=" * 65)

log_T = np.log(np.array(T_values, dtype=float))
print(f"\n{'Sector':<8} {'Level':<6} {'α (decay)':>10} {'R ratio Q/L':>12} {'Δ ratio':>10}")
print("-" * 50)
for lev in range(4):
    for pname in ['QUARK', 'LEPTON']:
        vals = np.array([convergence[T][pname][lev] for T in T_values])
        slope = np.polyfit(log_T, np.log(vals), 1)[0]
        alpha = -slope
        label = f"{pname[0]}"
        if pname == 'QUARK':
            qr = [convergence[T]['QUARK'][lev] / convergence[T]['LEPTON'][lev] for T in T_values]
            delta = abs(qr[2] - qr[1])
            print(f"  {label:<6} R{lev+1:<4} {alpha:>10.4f}   {qr[0]:>5.3f}→{qr[2]:>5.3f}  {delta:>10.4f}")
        else:
            print(f"  {label:<6} R{lev+1:<4} {alpha:>10.4f}")

# Key question: do decay exponents have number-theoretic values?
print(f"\n--- Number-theoretic candidate check for α values ---")
from sympy import totient, reduced_totient
candidates = {
    '1/2': 0.5,
    'ln2': np.log(2),
    'ln3': np.log(3), 
    'ln5': np.log(5),
    'ln7': np.log(7),
    '1/ln(210)': 1/np.log(210),
    'ρ': RHO,
    '2ρ': 2*RHO,
    'ρ²': RHO**2,
}
for pname in ['QUARK', 'LEPTON']:
    print(f"\n{pname}:")
    for lev in range(4):
        vals = np.array([convergence[T][pname][lev] for T in T_values])
        alpha = -np.polyfit(log_T, np.log(vals), 1)[0]
        best_name, best_val, best_dev = None, None, 999
        for cname, cval in candidates.items():
            dev = abs(alpha - cval) / cval
            if dev < best_dev:
                best_name, best_val, best_dev = cname, cval, dev
        print(f"  R{lev+1}: α = {alpha:.4f}, closest = {best_name} ({best_val:.4f}), dev = {best_dev:.1%}")

COMPACT T-SCALING SUMMARY

Sector   Level   α (decay)  R ratio Q/L    Δ ratio
--------------------------------------------------
  Q      R1        0.4991   5.578→4.373      0.4511
  L      R1        0.3236
  Q      R2        0.4973   3.414→2.050      0.5330
  L      R2        0.1294
  Q      R3        0.4716   1.405→0.899      0.1913
  L      R3        0.1491
  Q      R4        0.2068   0.800→0.889      0.0480
  L      R4        0.2837

--- Number-theoretic candidate check for α values ---

QUARK:
  R1: α = 0.4991, closest = 1/2 (0.5000), dev = 0.2%
  R2: α = 0.4973, closest = 1/2 (0.5000), dev = 0.5%
  R3: α = 0.4716, closest = 1/2 (0.5000), dev = 5.7%
  R4: α = 0.2068, closest = 1/ln(210) (0.1870), dev = 10.6%

LEPTON:
  R1: α = 0.3236, closest = 1/2 (0.5000), dev = 35.3%
  R2: α = 0.1294, closest = 2ρ (0.1380), dev = 6.2%
  R3: α = 0.1491, closest = 2ρ (0.1380), dev = 8.0%
  R4: α = 0.2837, closest = 1/2 (0.5000), dev = 43.3%


## Cell 4: Windowed R Analysis — Is There a Steady State?

The cumulative RMS decreases because it includes the transient from t=0.
If the system reaches quasi-equilibrium, the **local** R (measured in time windows
[t, t+W]) should stabilize.

Strategy: Run a long trajectory (T=30000), divide into windows of width W=3000,
compute R per window. If the windowed R converges, THAT is the physical value.
If it keeps decreasing, the system genuinely has no equilibrium and the R approach
needs fundamental reconsideration.

In [6]:
# ── Windowed R: time-local covering residuals ──
# Run long trajectory, compute R in non-overlapping windows
# FIX: use crossing NUMBER (ci) for CRT, not array index (crossings[ci])

T_TOTAL = 30000
W = 3000  # window width
N_BR = 50
np.random.seed(42)
sample = [ALL_BRANCHES[i] for i in np.random.choice(len(ALL_BRANCHES), N_BR, replace=False)]

ode = make_ode(EPS, EPS)

# Accumulate per-window, per-sector data
n_windows = T_TOTAL // W
# window_accum[win][a3_idx][a7_star][level] = [sum_sq, count]
window_accum = {}
for w in range(n_windows):
    window_accum[w] = {}
    for a3i in range(2):
        window_accum[w][a3i] = {}
        for a7s in range(6):
            window_accum[w][a3i][a7s] = {lev: [0.0, 0] for lev in range(4)}

total_used = 0
total_skipped = 0

for ib, br in enumerate(sample):
    theta0 = branch_ic(br)
    n_pts = int(T_TOTAL * 300)
    sol = solve_ivp(ode, [0, T_TOTAL], theta0,
                    t_eval=np.linspace(0, T_TOTAL, n_pts),
                    method='RK45', rtol=1e-12, atol=1e-14)
    
    th0_mod = np.mod(sol.y[0], 2*np.pi)
    crossings = np.where(np.diff(th0_mod) < -np.pi)[0]
    t_cross = sol.t[crossings]
    sections = sol.y[:, crossings]
    
    # Compute R at each crossing
    R_cross = np.zeros((4, len(crossings)))
    for k in range(4):
        p = PRIMES[k]
        raw = p * sections[k+1] - sections[k]
        R_cross[k] = np.mod(raw, 2*np.pi)
        R_cross[k][R_cross[k] > np.pi] -= 2*np.pi
    
    # Bin crossings into windows
    for ci in range(len(crossings)):
        tc = t_cross[ci]
        w_idx = min(int(tc // W), n_windows - 1)
        i = ci  # FIXED: crossing NUMBER, not array index
        if gcd(i, N) != 1:
            total_skipped += 1
            continue
        a3_raw, a5_raw, a7_raw = i % 3, i % 5, i % 7
        if a3_raw == 0 or a5_raw == 0 or a7_raw == 0:
            total_skipped += 1
            continue
        a5_idx = DLOG[5][a5_raw]
        if a5_idx != 0:
            total_skipped += 1
            continue  # physical sector only
        a3_idx = DLOG[3][a3_raw]
        a7_star = DLOG[7][a7_raw]
        for level in range(4):
            bucket = window_accum[w_idx][a3_idx][a7_star][level]
            bucket[0] += R_cross[level][ci] ** 2
            bucket[1] += 1
        total_used += 1
    
    if (ib + 1) % 10 == 0:
        print(f"  Branch {ib+1}/{N_BR}, crossings so far: used={total_used}, skipped={total_skipped}")

print(f"\nTotal: {total_used} crossings used, {total_skipped} skipped")

# Compute windowed CP-pair ratios
cp_pairs = {'LEPTON': (0, 1, 5), 'QUARK': (1, 4, 2)}
win_ratios = {pname: {lev: [] for lev in range(4)} for pname in cp_pairs}

for w in range(n_windows):
    for pname, (a3i, a7_g1, a7_g2) in cp_pairs.items():
        for lev in range(4):
            sq1, cnt1 = window_accum[w][a3i][a7_g1][lev]
            sq2, cnt2 = window_accum[w][a3i][a7_g2][lev]
            if cnt1 > 0 and cnt2 > 0:
                v1 = np.sqrt(sq1 / cnt1)
                v2 = np.sqrt(sq2 / cnt2)
                r = v1 / v2 if v2 > 0 else np.nan
            else:
                r = np.nan
            win_ratios[pname][lev].append(r)

# Display
print(f"\n{'='*80}")
print(f"WINDOWED CP-PAIR RATIOS (W={W}, {n_windows} windows)")
print(f"{'='*80}")
for pname in ['QUARK', 'LEPTON']:
    print(f"\n{pname}:")
    print(f"  {'Window':>10}  {'R₁':>10}  {'R₂':>10}  {'R₃':>10}  {'R₄':>10}")
    print(f"  {'-'*52}")
    for w in range(n_windows):
        t_start = w * W
        t_end = (w + 1) * W
        vals = [win_ratios[pname][lev][w] for lev in range(4)]
        label = f"{t_start//1000}k-{t_end//1000}k"
        print(f"  {label:>10}  {vals[0]:>10.4f}  {vals[1]:>10.4f}  {vals[2]:>10.4f}  {vals[3]:>10.4f}")
    # Check: is R₄ stabilizing?
    r4_late = [win_ratios[pname][3][w] for w in range(n_windows // 2, n_windows)]
    r4_mean = np.nanmean(r4_late)
    r4_std = np.nanstd(r4_late)
    print(f"  R₄ late-half: mean={r4_mean:.4f} ± {r4_std:.4f} ({r4_std/r4_mean*100:.1f}% CV)")

  Branch 10/50, crossings so far: used=17140, skipped=282855
  Branch 20/50, crossings so far: used=34280, skipped=565711
  Branch 30/50, crossings so far: used=51420, skipped=848568
  Branch 40/50, crossings so far: used=68560, skipped=1131424
  Branch 50/50, crossings so far: used=85700, skipped=1414281

Total: 85700 crossings used, 1414281 skipped

WINDOWED CP-PAIR RATIOS (W=3000, 10 windows)

QUARK:
      Window          R₁          R₂          R₃          R₄
  ----------------------------------------------------
       0k-3k     46.4591     25.5033      7.6630      1.7033
       3k-6k      1.0000      1.0000      1.0000      1.0000
       6k-9k      1.0000      1.0000      1.0000      1.0000
      9k-12k      1.0000      1.0000      1.0000      1.0000
     12k-15k      1.0000      1.0000      1.0000      1.0000
     15k-18k      1.0000      1.0000      1.0000      1.0000
     18k-21k      1.0000      1.0000      1.0000      1.0000
     21k-24k      1.0000      1.0000      1.0000  

In [8]:
# ── Ultra-compact windowed summary ──
for pname in ['QUARK', 'LEPTON']:
    vals4 = win_ratios[pname][3]
    vals3 = win_ratios[pname][2]
    nan4 = sum(1 for v in vals4 if np.isnan(v))
    nan3 = sum(1 for v in vals3 if np.isnan(v))
    valid4 = [v for v in vals4 if not np.isnan(v)]
    valid3 = [v for v in vals3 if not np.isnan(v)]
    print(f"{pname} R4: {nan4} NaN / {len(vals4)} windows")
    if valid4:
        print(f"  first={valid4[0]:.4f} last={valid4[-1]:.4f} trend={((valid4[-1]-valid4[0])/valid4[0]*100):+.1f}%")
        print(f"  late-half mean={np.nanmean(vals4[5:]):.4f} +/- {np.nanstd(vals4[5:]):.4f}")
    print(f"{pname} R3: {nan3} NaN / {len(vals3)} windows")
    if valid3:
        print(f"  first={valid3[0]:.4f} last={valid3[-1]:.4f}")
    print()
print(f"Total crossings: used={total_used}, skipped={total_skipped}")

QUARK R4: 0 NaN / 10 windows
  first=1.7033 last=1.0000 trend=-41.3%
  late-half mean=1.0000 +/- 0.0000
QUARK R3: 0 NaN / 10 windows
  first=7.6630 last=1.0000

LEPTON R4: 0 NaN / 10 windows
  first=2.3474 last=1.0000 trend=-57.4%
  late-half mean=1.0000 +/- 0.0000
LEPTON R3: 0 NaN / 10 windows
  first=4.2894 last=1.0000

Total crossings: used=85700, skipped=1414281


## Phase 2: Fine-Grained R(t) Trajectory

The coarse windowed analysis revealed that **all CP-pair ratios converge to 1.0** at late times.
Both partners relax toward the exact solenoid at the same rate, so ratio approaches 1.

But NB70-73 used T=5000 and got accurate mass predictions. This means:
- The **transient regime** contains the physics
- There may be a preferred timescale where R(t) encodes mass ratios
- The decay RATE may itself have algebraic structure

**Strategy**: Use fine windows (W=200) over T=10000 to map the complete R(t) trajectory
at all four levels, then fit decay models to identify timescales.

In [9]:
# ── Fine-grained R(t): W=200 windows over T=10000 ──
T_FINE = 10000
W_FINE = 200
N_BR_FINE = 50
np.random.seed(42)
sample_fine = [ALL_BRANCHES[i] for i in np.random.choice(len(ALL_BRANCHES), N_BR_FINE, replace=False)]

n_win_fine = T_FINE // W_FINE  # 50 windows

# Accumulator: [win][a3_idx][a7_star][level] = [sum_sq, count]
fine_accum = {}
for w in range(n_win_fine):
    fine_accum[w] = {}
    for a3i in range(2):
        fine_accum[w][a3i] = {}
        for a7s in range(6):
            fine_accum[w][a3i][a7s] = {lev: [0.0, 0] for lev in range(4)}

# Also track individual RMS per CP pair per window (for both partners separately)
# rms_accum[win][a3_idx][a7_star][level] = [sum_sq, count]
# (same as fine_accum, just using it for individual partner RMS)

ode_fine = make_ode(EPS, EPS)
for ib, br in enumerate(sample_fine):
    theta0 = branch_ic(br)
    n_pts = int(T_FINE * 300)
    sol = solve_ivp(ode_fine, [0, T_FINE], theta0,
                    t_eval=np.linspace(0, T_FINE, n_pts),
                    method='RK45', rtol=1e-12, atol=1e-14)
    
    th0_mod = np.mod(sol.y[0], 2*np.pi)
    crossings = np.where(np.diff(th0_mod) < -np.pi)[0]
    t_cross = sol.t[crossings]
    sections = sol.y[:, crossings]
    
    R_cross = np.zeros((4, len(crossings)))
    for k in range(4):
        p = PRIMES[k]
        raw = p * sections[k+1] - sections[k]
        R_cross[k] = np.mod(raw, 2*np.pi)
        R_cross[k][R_cross[k] > np.pi] -= 2*np.pi
    
    for ci in range(len(crossings)):
        tc = t_cross[ci]
        w_idx = min(int(tc // W_FINE), n_win_fine - 1)
        i = ci  # crossing number
        if gcd(i, N) != 1:
            continue
        a3_raw, a5_raw, a7_raw = i % 3, i % 5, i % 7
        if a3_raw == 0 or a5_raw == 0 or a7_raw == 0:
            continue
        a5_idx = DLOG[5][a5_raw]
        if a5_idx != 0:
            continue
        a3_idx = DLOG[3][a3_raw]
        a7_star = DLOG[7][a7_raw]
        for level in range(4):
            bucket = fine_accum[w_idx][a3_idx][a7_star][level]
            bucket[0] += R_cross[level][ci] ** 2
            bucket[1] += 1
    
    if (ib + 1) % 10 == 0:
        print(f"  Branch {ib+1}/{N_BR_FINE}")

# Compute fine-grained CP-pair ratios AND individual RMS values
cp_pairs_fine = {'LEPTON': (0, 1, 5), 'QUARK': (1, 4, 2)}
fine_ratios = {pn: {lv: [] for lv in range(4)} for pn in cp_pairs_fine}
fine_rms = {}  # fine_rms[pname][partner][level] = list over windows
for pn in cp_pairs_fine:
    fine_rms[pn] = {'g1': {lv: [] for lv in range(4)}, 'g2': {lv: [] for lv in range(4)}}

for w in range(n_win_fine):
    for pn, (a3i, g1, g2) in cp_pairs_fine.items():
        for lv in range(4):
            sq1, c1 = fine_accum[w][a3i][g1][lv]
            sq2, c2 = fine_accum[w][a3i][g2][lv]
            v1 = np.sqrt(sq1 / c1) if c1 > 0 else np.nan
            v2 = np.sqrt(sq2 / c2) if c2 > 0 else np.nan
            fine_rms[pn]['g1'][lv].append(v1)
            fine_rms[pn]['g2'][lv].append(v2)
            if c1 > 0 and c2 > 0 and v2 > 0:
                fine_ratios[pn][lv].append(v1 / v2)
            else:
                fine_ratios[pn][lv].append(np.nan)

# Display compact trajectory for R4
t_centers = [(w + 0.5) * W_FINE for w in range(n_win_fine)]
print("FINE-GRAINED R4 TRAJECTORY (W=200, 50 windows)")
print("=" * 70)
print(f"  {'t_mid':>6}  {'Q_R4':>8}  {'L_R4':>8}  {'Q_rms1':>8}  {'Q_rms2':>8}  {'L_rms1':>8}  {'L_rms2':>8}")
print(f"  {'-'*62}")
for w in range(n_win_fine):
    t = t_centers[w]
    qr4 = fine_ratios['QUARK'][3][w]
    lr4 = fine_ratios['LEPTON'][3][w]
    qr1 = fine_rms['QUARK']['g1'][3][w]
    qr2 = fine_rms['QUARK']['g2'][3][w]
    lr1 = fine_rms['LEPTON']['g1'][3][w]
    lr2 = fine_rms['LEPTON']['g2'][3][w]
    print(f"  {t:6.0f}  {qr4:8.4f}  {lr4:8.4f}  {qr1:8.5f}  {qr2:8.5f}  {lr1:8.5f}  {lr2:8.5f}")

# Key cross-check: what does cumulative R4 equal at T=5000?
print(f"\n\nCUMULATIVE R4 at key times:")
for T_check in [1000, 2000, 3000, 5000, 7000, 10000]:
    w_end = T_check // W_FINE
    # Cumulative = pool all windows up to w_end
    for pn, (a3i, g1, g2) in cp_pairs_fine.items():
        sq1_cum, c1_cum, sq2_cum, c2_cum = 0, 0, 0, 0
        for w in range(w_end):
            s1, n1 = fine_accum[w][a3i][g1][3]
            s2, n2 = fine_accum[w][a3i][g2][3]
            sq1_cum += s1; c1_cum += n1
            sq2_cum += s2; c2_cum += n2
        v1 = np.sqrt(sq1_cum / c1_cum) if c1_cum > 0 else 0
        v2 = np.sqrt(sq2_cum / c2_cum) if c2_cum > 0 else 0
        ratio = v1 / v2 if v2 > 0 else 0
        print(f"  T={T_check:>5}, {pn:>6} R4_cum = {ratio:.6f}  (rms1={v1:.6f}, rms2={v2:.6f}, n1={c1_cum}, n2={c2_cum})")

  Branch 10/50
  Branch 20/50
  Branch 30/50
  Branch 40/50
  Branch 50/50
FINE-GRAINED R4 TRAJECTORY (W=200, 50 windows)
   t_mid      Q_R4      L_R4    Q_rms1    Q_rms2    L_rms1    L_rms2
  --------------------------------------------------------------
     100    5.4292    5.8717   1.69072   0.31141   2.09265   0.35640
     300       nan    1.0000   0.31120       nan   0.24048   0.24049
     500    1.0000    1.0000   0.31115   0.31115   0.24049   0.24049
     700    1.0000    1.0000   0.31116   0.31116   0.24049   0.24049
     900    1.0000    1.0000   0.31116   0.31116   0.24049   0.24048
    1100    1.0000    1.0000   0.31116   0.31116   0.24048   0.24048
    1300    1.0000    1.0000   0.31116   0.31116   0.24048   0.24048
    1500    1.0000    1.0000   0.31116   0.31116   0.24048   0.24048
    1700    1.0000    1.0000   0.31116   0.31116   0.24048   0.24048
    1900    1.0000    1.0000   0.31116   0.31116   0.24047   0.24047
    2100    1.0000    1.0000   0.31117   0.31117   0.2

In [10]:
# ── Compact: cumulative R4 at key times + NB73 comparison ──
print("CUMULATIVE R4 vs NB73 VALUES")
print("=" * 65)
# NB73 values (T=5000, 50 branches, seed=42)
NB73_R4_q = 1.4794
NB73_R4_l = 1.9795

for T_check in [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 10000]:
    w_end = T_check // W_FINE
    row = f"  T={T_check:>5}: "
    for pn, (a3i, g1, g2) in cp_pairs_fine.items():
        sq1_cum, c1_cum, sq2_cum, c2_cum = 0, 0, 0, 0
        for w in range(w_end):
            s1, n1 = fine_accum[w][a3i][g1][3]
            s2, n2 = fine_accum[w][a3i][g2][3]
            sq1_cum += s1; c1_cum += n1
            sq2_cum += s2; c2_cum += n2
        v1 = np.sqrt(sq1_cum / c1_cum) if c1_cum > 0 else 0
        v2 = np.sqrt(sq2_cum / c2_cum) if c2_cum > 0 else 0
        ratio = v1 / v2 if v2 > 0 else 0
        row += f" {pn[:1]}={ratio:.4f}"
    print(row)

print(f"\n  NB73:        Q={NB73_R4_q:.4f}  L={NB73_R4_l:.4f}")

# Check: do windowed ratios monotonically decrease or is there a plateau?
print(f"\nPER-WINDOW R4 (not cumulative):")
print(f"  {'win':>4} {'t_mid':>6} {'Q_R4':>8} {'L_R4':>8}")
for w in [0,1,2,3,4,5,9,14,19,24,29,39,49]:
    if w < n_win_fine:
        t = (w + 0.5) * W_FINE
        qr = fine_ratios['QUARK'][3][w]
        lr = fine_ratios['LEPTON'][3][w]
        print(f"  {w:>4} {t:6.0f} {qr:8.4f} {lr:8.4f}")

CUMULATIVE R4 vs NB73 VALUES
  T= 1000:  L=3.5869 Q=2.5889
  T= 2000:  L=2.7508 Q=1.9626
  T= 3000:  L=2.3475 Q=1.7033
  T= 4000:  L=2.1543 Q=1.5815
  T= 5000:  L=1.9795 Q=1.4793
  T= 6000:  L=1.8535 Q=1.4083
  T= 7000:  L=1.7577 Q=1.3560
  T= 8000:  L=1.6960 Q=1.3158
  T=10000:  L=1.5796 Q=1.2626

  NB73:        Q=1.4794  L=1.9795

PER-WINDOW R4 (not cumulative):
   win  t_mid     Q_R4     L_R4
     0    100   5.4292   5.8717
     1    300      nan   1.0000
     2    500   1.0000   1.0000
     3    700   1.0000   1.0000
     4    900   1.0000   1.0000
     5   1100   1.0000   1.0000
     9   1900   1.0000   1.0000
    14   2900   1.0000      nan
    19   3900      nan   1.0000
    24   4900   1.0000   1.0000
    29   5900   1.0000   1.0000
    39   7900   1.0000   1.0000
    49   9900   1.0000   1.0000


In [12]:
# ── Diagnose: why is per-window ratio exactly 1.0? ──
# QUARK: a3=1, a7*=4 vs a7*=2 at level=3
print("QUARK CP bins (a3=1, a7*=4 vs 2), Level 3, first 6 windows:")
for w in range(6):
    sq1, n1 = fine_accum[w][1][4][3]
    sq2, n2 = fine_accum[w][1][2][3]
    rms1 = np.sqrt(sq1/n1) if n1 > 0 else 0
    rms2 = np.sqrt(sq2/n2) if n2 > 0 else 0
    rat = rms1/rms2 if rms2 > 0 else float('nan')
    print(f"  w{w}: n1={n1:>5} n2={n2:>5} rms1={rms1:.6f} rms2={rms2:.6f} R={rat:.6f}")

print("\nLEPTON CP bins (a3=0, a7*=1 vs 5), Level 3, first 6 windows:")
for w in range(6):
    sq1, n1 = fine_accum[w][0][1][3]
    sq2, n2 = fine_accum[w][0][5][3]
    rms1 = np.sqrt(sq1/n1) if n1 > 0 else 0
    rms2 = np.sqrt(sq2/n2) if n2 > 0 else 0
    rat = rms1/rms2 if rms2 > 0 else float('nan')
    print(f"  w{w}: n1={n1:>5} n2={n2:>5} rms1={rms1:.6f} rms2={rms2:.6f} R={rat:.6f}")

# Stats
cxpw = W_FINE / (2*np.pi)
print(f"\nCrossings per window per branch: ~{cxpw:.1f}")
print(f"Per (a3,a7*) bin per window (~48/210/12): ~{cxpw*50*48/210/12:.1f}")

QUARK CP bins (a3=1, a7*=4 vs 2), Level 3, first 6 windows:
  w0: n1=   50 n2=   50 rms1=1.690716 rms2=0.311412 R=5.429202
  w1: n1=   50 n2=    0 rms1=0.311196 rms2=0.000000 R=nan
  w2: n1=   50 n2=   50 rms1=0.311155 rms2=0.311154 R=1.000001
  w3: n1=   50 n2=   50 rms1=0.311156 rms2=0.311156 R=1.000001
  w4: n1=   50 n2=   50 rms1=0.311157 rms2=0.311157 R=1.000001
  w5: n1=   50 n2=   50 rms1=0.311159 rms2=0.311158 R=1.000001

LEPTON CP bins (a3=0, a7*=1 vs 5), Level 3, first 6 windows:
  w0: n1=   50 n2=   50 rms1=2.092652 rms2=0.356398 R=5.871675
  w1: n1=   50 n2=   50 rms1=0.240480 rms2=0.240490 R=0.999958
  w2: n1=   50 n2=   50 rms1=0.240490 rms2=0.240489 R=1.000001
  w3: n1=   50 n2=   50 rms1=0.240487 rms2=0.240487 R=1.000001
  w4: n1=   50 n2=   50 rms1=0.240485 rms2=0.240485 R=1.000001
  w5: n1=   50 n2=   50 rms1=0.240483 rms2=0.240482 R=1.000001

Crossings per window per branch: ~31.8
Per (a3,a7*) bin per window (~48/210/12): ~30.3


### Key Finding: CP Asymmetry is Initial-Condition Geometry

The per-window diagnostic reveals:
- **Window 0**: Massive asymmetry (R~5.4 quark, R~5.9 lepton) 
- **All subsequent windows**: R = 1.000001 (identical RMS for both CP partners)
- Each bin has exactly 50 counts per window = 1 per branch

The cumulative R(T) that NB70-73 used is therefore a **dilution curve**:

$$R_{\text{cum}}^2(T) = \frac{A + B \cdot n_w}{C + B \cdot n_w}$$

where A = rms_01^2, C = rms_02^2, B = rms_late^2, n_w = T/W - 1.

The physics lives in the **initial-condition asymmetry** (A vs C),
not in the ODE dynamics at finite T.

In [14]:
# ── Verify dilution model + extract initial asymmetry ──

# From window 0 and late windows
rms_q1_0 = np.sqrt(fine_accum[0][1][4][3][0] / fine_accum[0][1][4][3][1])
rms_q2_0 = np.sqrt(fine_accum[0][1][2][3][0] / fine_accum[0][1][2][3][1])
rms_q_late = np.sqrt(fine_accum[2][1][4][3][0] / fine_accum[2][1][4][3][1])
rms_l1_0 = np.sqrt(fine_accum[0][0][1][3][0] / fine_accum[0][0][1][3][1])
rms_l2_0 = np.sqrt(fine_accum[0][0][5][3][0] / fine_accum[0][0][5][3][1])
rms_l_late = np.sqrt(fine_accum[2][0][1][3][0] / fine_accum[2][0][1][3][1])

R0_q = rms_q1_0 / rms_q2_0
R0_l = rms_l1_0 / rms_l2_0

print(f"R0_quark  = {R0_q:.8f}  (rms: {rms_q1_0:.6f} / {rms_q2_0:.6f})")
print(f"R0_lepton = {R0_l:.8f}  (rms: {rms_l1_0:.6f} / {rms_l2_0:.6f})")
print(f"late rms:  Q={rms_q_late:.6f}  L={rms_l_late:.6f}")

# Dilution model check at T=5000
A_q, C_q, B_q = rms_q1_0**2, rms_q2_0**2, rms_q_late**2
A_l, C_l, B_l = rms_l1_0**2, rms_l2_0**2, rms_l_late**2
nw1 = 5000 // W_FINE - 1  # 24
R_q_model = np.sqrt((A_q + B_q*nw1) / (C_q + B_q*nw1))
R_l_model = np.sqrt((A_l + B_l*nw1) / (C_l + B_l*nw1))
print(f"\nModel T=5000: Q={R_q_model:.6f} L={R_l_model:.6f}")
print(f"NB73  T=5000: Q=1.479400 L=1.979500")

R0_quark  = 5.42920162  (rms: 1.690716 / 0.311412)
R0_lepton = 5.87167522  (rms: 2.092652 / 0.356398)
late rms:  Q=0.311155  L=0.240490

Model T=5000: Q=1.463166 L=1.951048
NB73  T=5000: Q=1.479400 L=1.979500


In [15]:
# ── Seed-independence test: R₀ across different branch samples ──
# If R₀ is seed-dependent, the mass predictions are artifacts

def quick_R0(seed, n_br=50, T_val=1000):
    """Compute window-0 CP-pair ratios for a given seed."""
    np.random.seed(seed)
    samp = [ALL_BRANCHES[i] for i in np.random.choice(len(ALL_BRANCHES), n_br, replace=False)]
    ode = make_ode(EPS, EPS)
    acc = {}
    for a3i in range(2):
        acc[a3i] = {}
        for a7s in range(6):
            acc[a3i][a7s] = [0.0, 0]  # level 3 only
    
    for br in samp:
        theta0 = branch_ic(br)
        n_pts = int(T_val * 300)
        sol = solve_ivp(ode, [0, T_val], theta0,
                        t_eval=np.linspace(0, T_val, n_pts),
                        method='RK45', rtol=1e-12, atol=1e-14)
        th0_mod = np.mod(sol.y[0], 2*np.pi)
        crossings = np.where(np.diff(th0_mod) < -np.pi)[0]
        sections = sol.y[:, crossings]
        R4 = PRIMES[3] * sections[4] - sections[3]
        R4 = np.mod(R4, 2*np.pi)
        R4[R4 > np.pi] -= 2*np.pi
        
        for ci in range(len(crossings)):
            i = ci
            if gcd(i, N) != 1:
                continue
            a3_raw, a5_raw, a7_raw = i % 3, i % 5, i % 7
            if a3_raw == 0 or a5_raw == 0 or a7_raw == 0:
                continue
            if DLOG[5][a5_raw] != 0:
                continue
            a3_idx = DLOG[3][a3_raw]
            a7_star = DLOG[7][a7_raw]
            acc[a3_idx][a7_star][0] += R4[ci]**2
            acc[a3_idx][a7_star][1] += 1
    
    # Extract CP ratios
    result = {}
    for pn, (a3i, g1, g2) in [('Q', (1,4,2)), ('L', (0,1,5))]:
        sq1, n1 = acc[a3i][g1]
        sq2, n2 = acc[a3i][g2]
        rms1 = np.sqrt(sq1/n1) if n1 > 0 else 0
        rms2 = np.sqrt(sq2/n2) if n2 > 0 else 0
        result[pn] = rms1/rms2 if rms2 > 0 else 0
    return result

print("SEED-INDEPENDENCE TEST (T=1000, 50 branches)")
print("=" * 45)
for seed in [42, 99, 137, 271, 1000]:
    r = quick_R0(seed)
    print(f"  seed={seed:>4}: Q_R4={r['Q']:.6f}  L_R4={r['L']:.6f}")

# Also test with more branches
print(f"\nBRANCH COUNT CONVERGENCE (seed=42):")
for n_br in [10, 25, 50, 100]:
    r = quick_R0(42, n_br=n_br)
    print(f"  n_br={n_br:>3}: Q_R4={r['Q']:.6f}  L_R4={r['L']:.6f}")

SEED-INDEPENDENCE TEST (T=1000, 50 branches)
  seed=  42: Q_R4=2.588651  L_R4=3.587422
  seed=  99: Q_R4=2.763628  L_R4=3.678297
  seed= 137: Q_R4=2.676956  L_R4=3.781496
  seed= 271: Q_R4=2.696589  L_R4=3.774190
  seed=1000: Q_R4=2.882094  L_R4=3.531108

BRANCH COUNT CONVERGENCE (seed=42):
  n_br= 10: Q_R4=2.480354  L_R4=3.668991
  n_br= 25: Q_R4=2.515720  L_R4=3.809636
  n_br= 50: Q_R4=2.588651  L_R4=3.587422
  n_br=100: Q_R4=2.770219  L_R4=3.549203


In [18]:
# ── Population R₀: ALL branches, parallelized ──
# Previous version: (a) ran quick_R0 THEN full loop = double work, ~16 min
#                   (b) n_factor=300 = 300 output pts/crossing, only ~10 needed
#                   (c) serial loop, no parallelism
# Fix: single pass, n_factor=50, ThreadPoolExecutor (scipy releases GIL in C)

from concurrent.futures import ThreadPoolExecutor, as_completed
import time

n_all = len(ALL_BRANCHES)
T_POP = 5000
N_FACTOR_POP = 50  # 50 pts per base period (was 300 — 6x speedup)

def process_branch(br):
    """Integrate one branch, return per-CRT-bin R4² accumulations."""
    ode = make_ode(EPS, EPS)
    theta0 = branch_ic(br)
    n_pts = int(T_POP * N_FACTOR_POP)
    sol = solve_ivp(ode, [0, T_POP], theta0,
                    t_eval=np.linspace(0, T_POP, n_pts),
                    method='RK45', rtol=1e-12, atol=1e-14)
    th0_mod = np.mod(sol.y[0], 2*np.pi)
    crossings = np.where(np.diff(th0_mod) < -np.pi)[0]
    sections = sol.y[:, crossings]
    R4 = PRIMES[3] * sections[4] - sections[3]
    R4 = np.mod(R4, 2*np.pi)
    R4[R4 > np.pi] -= 2*np.pi
    
    local = {}  # (a3_idx, a7_star) -> [sum_sq, count]
    for ci in range(len(crossings)):
        i = ci
        if gcd(i, N) != 1:
            continue
        a3_raw, a5_raw, a7_raw = i % 3, i % 5, i % 7
        if a3_raw == 0 or a5_raw == 0 or a7_raw == 0:
            continue
        if DLOG[5][a5_raw] != 0:
            continue
        a3_idx = DLOG[3][a3_raw]
        a7_star = DLOG[7][a7_raw]
        key = (a3_idx, a7_star)
        if key not in local:
            local[key] = [0.0, 0]
        local[key][0] += R4[ci]**2
        local[key][1] += 1
    return local

print(f"Population run: {n_all} branches, T={T_POP}, n_factor={N_FACTOR_POP}")
t0 = time.time()

# Parallel execution
pop_acc = {}
for a3i in range(2):
    pop_acc[a3i] = {}
    for a7s in range(6):
        pop_acc[a3i][a7s] = {3: [0.0, 0]}

done = 0
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(process_branch, br): br for br in ALL_BRANCHES}
    for fut in as_completed(futures):
        local = fut.result()
        for (a3_idx, a7_star), (sq, cnt) in local.items():
            pop_acc[a3_idx][a7_star][3][0] += sq
            pop_acc[a3_idx][a7_star][3][1] += cnt
        done += 1
        if done % 30 == 0:
            elapsed = time.time() - t0
            rate = done / elapsed
            eta = (n_all - done) / rate
            print(f"  {done}/{n_all} branches ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

elapsed = time.time() - t0
print(f"\nCompleted in {elapsed:.1f}s ({elapsed/n_all:.2f}s per branch)")

# Population CP-pair ratios
print(f"\nPOPULATION R4 (ALL {n_all} branches, T={T_POP}):")
for pn, (a3i, g1, g2) in [('QUARK', (1,4,2)), ('LEPTON', (0,1,5))]:
    s1, n1 = pop_acc[a3i][g1][3]
    s2, n2 = pop_acc[a3i][g2][3]
    rms1 = np.sqrt(s1/n1) if n1 > 0 else 0
    rms2 = np.sqrt(s2/n2) if n2 > 0 else 0
    r = rms1/rms2 if rms2 > 0 else 0
    print(f"  {pn}: R4 = {r:.8f}  (n1={n1}, n2={n2})")
    print(f"    rms1={rms1:.8f}  rms2={rms2:.8f}")

Population run: 210 branches, T=5000, n_factor=50
  30/210 branches (184s elapsed, ~1104s remaining)
  60/210 branches (368s elapsed, ~920s remaining)
  90/210 branches (542s elapsed, ~722s remaining)
  120/210 branches (695s elapsed, ~522s remaining)
  150/210 branches (879s elapsed, ~352s remaining)
  180/210 branches (1046s elapsed, ~174s remaining)
  210/210 branches (1197s elapsed, ~0s remaining)

Completed in 1197.2s (5.70s per branch)

POPULATION R4 (ALL 210 branches, T=5000):
  QUARK: R4 = 1.53945200  (n1=5040, n2=4830)
    rms1=0.47882049  rms2=0.31103308
  LEPTON: R4 = 1.93972469  (n1=5040, n2=5040)
    rms1=0.47509416  rms2=0.24492865


In [17]:
# ── NB74 State Snapshot ──
print("=== NB74 CRITICAL NUMBERS ===")
print(f"R0_quark  = {R0_q:.8f}")
print(f"R0_lepton = {R0_l:.8f}")
print(f"NB73 R4_q = {NB73_R4_q}")
print(f"NB73 R4_l = {NB73_R4_l}")
print(f"Dilution model at T=5000: Q={R_q_model:.6f}  L={R_l_model:.6f}")
print(f"RMS window0 Q: g1={rms_q1_0:.6f} g2={rms_q2_0:.6f}")
print(f"RMS window0 L: g1={rms_l1_0:.6f} g2={rms_l2_0:.6f}")
print(f"RMS late Q: {rms_q_late:.6f}  L: {rms_l_late:.6f}")
print(f"\nFine-grained R4 per window (first 5):")
for w in range(5):
    qr = fine_ratios['QUARK'][3][w]
    lr = fine_ratios['LEPTON'][3][w]
    print(f"  w{w}: Q={qr:.6f} L={lr:.6f}")

=== NB74 CRITICAL NUMBERS ===
R0_quark  = 5.42920162
R0_lepton = 5.87167522
NB73 R4_q = 1.4794
NB73 R4_l = 1.9795
Dilution model at T=5000: Q=1.463166  L=1.951048
RMS window0 Q: g1=1.690716 g2=0.311412
RMS window0 L: g1=2.092652 g2=0.356398
RMS late Q: 0.311155  L: 0.240490

Fine-grained R4 per window (first 5):
  w0: Q=5.429202 L=5.871675
  w1: Q=nan L=0.999958
  w2: Q=1.000001 L=1.000001
  w3: Q=1.000001 L=1.000001
  w4: Q=1.000001 L=1.000001


## Phase 3: Population Window-0 Asymmetry

The fine-grained analysis revealed the **entire CP asymmetry** is in window 0 (first ~200 time units).
All later windows have R = 1.000. The cumulative R(T) from NB70-73 was a dilution curve.

**Threading analysis**: `ThreadPoolExecutor` gave no speedup because `solve_ivp` calls the Python ODE
function millions of times, each acquiring the GIL. True parallelism requires either:
- `ProcessPoolExecutor` (tricky in Jupyter/Windows)
- numba-compiled ODE RHS (releases GIL)
- For now, serial is fine for short T values

**Strategy**: Run ALL 210 branches at T=200 only (window 0) to get the definitive population R₀.
This is 200/5000 = 4% of the full integration → ~0.25s per branch → ~50s total.

In [19]:
# ── Population Window-0 R₀: ALL 210 branches, T=200 (serial, ~50s) ──
import time

T_W0 = 200
n_all = len(ALL_BRANCHES)
print(f"Population R₀: {n_all} branches, T={T_W0}")

# Accumulator for ALL levels, not just level 3
w0_acc = {}
for a3i in range(2):
    w0_acc[a3i] = {}
    for a7s in range(6):
        w0_acc[a3i][a7s] = {lev: [0.0, 0] for lev in range(4)}

t0 = time.time()
for ib, br in enumerate(ALL_BRANCHES):
    theta0 = branch_ic(br)
    n_pts = int(T_W0 * 100)  # 100 pts/period — sufficient for crossing detection
    ode = make_ode(EPS, EPS)
    sol = solve_ivp(ode, [0, T_W0], theta0,
                    t_eval=np.linspace(0, T_W0, n_pts),
                    method='RK45', rtol=1e-12, atol=1e-14)
    th0_mod = np.mod(sol.y[0], 2*np.pi)
    crossings = np.where(np.diff(th0_mod) < -np.pi)[0]
    sections = sol.y[:, crossings]
    
    R_cross = np.zeros((4, len(crossings)))
    for k in range(4):
        p = PRIMES[k]
        raw = p * sections[k+1] - sections[k]
        R_cross[k] = np.mod(raw, 2*np.pi)
        R_cross[k][R_cross[k] > np.pi] -= 2*np.pi
    
    for ci in range(len(crossings)):
        i = ci  # crossing number
        if gcd(i, N) != 1:
            continue
        a3_raw, a5_raw, a7_raw = i % 3, i % 5, i % 7
        if a3_raw == 0 or a5_raw == 0 or a7_raw == 0:
            continue
        if DLOG[5][a5_raw] != 0:
            continue
        a3_idx = DLOG[3][a3_raw]
        a7_star = DLOG[7][a7_raw]
        for level in range(4):
            w0_acc[a3_idx][a7_star][level][0] += R_cross[level][ci] ** 2
            w0_acc[a3_idx][a7_star][level][1] += 1
    
    if (ib + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f"  {ib+1}/{n_all} ({elapsed:.1f}s)")

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s ({elapsed/n_all:.3f}s/branch)")

# Population R₀ at all 4 levels
print(f"\n{'='*80}")
print(f"POPULATION R₀ (all {n_all} branches, T={T_W0}, window-0 only)")
print(f"{'='*80}")
cp_pairs_w0 = {'QUARK': (1, 4, 2), 'LEPTON': (0, 1, 5)}
pop_R0 = {}
for pn, (a3i, g1, g2) in cp_pairs_w0.items():
    print(f"\n{pn} (a3={a3i}, a7*={g1} vs {g2}):")
    for lev in range(4):
        s1, n1 = w0_acc[a3i][g1][lev]
        s2, n2 = w0_acc[a3i][g2][lev]
        rms1 = np.sqrt(s1/n1) if n1 > 0 else 0
        rms2 = np.sqrt(s2/n2) if n2 > 0 else 0
        ratio = rms1/rms2 if rms2 > 0 else 0
        print(f"  R₀[{lev}] = {ratio:.8f}  (rms1={rms1:.8f}, rms2={rms2:.8f}, n1={n1}, n2={n2})")
        pop_R0[(pn, lev)] = ratio

# Key comparison
print(f"\n--- R₄ comparison ---")
print(f"Pop R₀ (window-0, T=200): Q={pop_R0[('QUARK',3)]:.6f}  L={pop_R0[('LEPTON',3)]:.6f}")
print(f"50-branch R₀ (seed=42):   Q={R0_q:.6f}  L={R0_l:.6f}")
print(f"Pop cumulative (T=5000):   Q=1.539452  L=1.939725")
print(f"NB73 (50br, T=5000):       Q=1.479400  L=1.979500")

Population R₀: 210 branches, T=200
  50/210 (11.1s)
  100/210 (22.2s)
  150/210 (33.2s)
  200/210 (44.0s)
Done in 46.1s (0.220s/branch)

POPULATION R₀ (all 210 branches, T=200, window-0 only)

QUARK (a3=1, a7*=4 vs 2):
  R₀[0] = 176.55995450  (rms1=1.93794432, rms2=0.01097613, n1=210, n2=210)
  R₀[1] = 97.34585776  (rms1=1.59842279, rms2=0.01642004, n1=210, n2=210)
  R₀[2] = 29.92247177  (rms1=1.74013667, rms2=0.05815484, n1=210, n2=210)
  R₀[3] = 5.81269217  (rms1=1.81031540, rms2=0.31144182, n1=210, n2=210)

LEPTON (a3=0, a7*=1 vs 5):
  R₀[0] = 8.83588634  (rms1=0.48168819, rms2=0.05451498, n1=210, n2=210)
  R₀[1] = 6.23422295  (rms1=1.25670342, rms2=0.20158140, n1=210, n2=210)
  R₀[2] = 4.69092384  (rms1=2.10015190, rms2=0.44770539, n1=210, n2=210)
  R₀[3] = 6.17597016  (rms1=2.02154871, rms2=0.32732488, n1=210, n2=210)

--- R₄ comparison ---
Pop R₀ (window-0, T=200): Q=5.812692  L=6.175970
50-branch R₀ (seed=42):   Q=5.429202  L=5.871675
Pop cumulative (T=5000):   Q=1.539452  L=1.9

In [20]:
# ── Number-theoretic scan of R₀ values ──
from fractions import Fraction

# Population R₀ values (level 3 = R₄, the mass-relevant level)
R0_q_pop = pop_R0[('QUARK', 3)]    # 5.81269
R0_l_pop = pop_R0[('LEPTON', 3)]   # 6.17597

# Key derived quantities
R0_ratio = R0_l_pop / R0_q_pop
R0_product = R0_l_pop * R0_q_pop
R0_sum = R0_l_pop + R0_q_pop
R0_diff = R0_l_pop - R0_q_pop
R0_q_sq = R0_q_pop ** 2
R0_l_sq = R0_l_pop ** 2

print("POPULATION R₀ NUMBER-THEORETIC SCAN")
print("=" * 70)
print(f"R₀_q = {R0_q_pop:.8f}")
print(f"R₀_l = {R0_l_pop:.8f}")
print(f"L/Q  = {R0_ratio:.8f}")
print(f"L×Q  = {R0_product:.8f}")
print(f"L+Q  = {R0_sum:.8f}")
print(f"L-Q  = {R0_diff:.8f}")
print(f"Q²   = {R0_q_sq:.8f}")
print(f"L²   = {R0_l_sq:.8f}")

# Candidates from {2,3,5,7} arithmetic
candidates_scan = {
    '2π': 2*np.pi,
    'π²': np.pi**2,
    'P₄': 210,
    'φ(210)': 48,
    'λ(210)': 12,
    'p₄²': 49,
    'p₄': 7,
    'p₃': 5,
    'p₂': 3,
    'p₄/p₂': 7/3,
    'p₄/p₃': 7/5,
    'p₃/p₂': 5/3,
    '√210': np.sqrt(210),
    '√(210)/2π': np.sqrt(210)/(2*np.pi),
    '210^(1/4)': 210**0.25,
    'φ/2π': 48/(2*np.pi),
    'λ/2π': 12/(2*np.pi),
    '49/2π': 49/(2*np.pi),
    '48/2π': 48/(2*np.pi),
    'φ/d': 3.0,
    'd(210)': 16,
    '35': 35,
    '42': 42,
    '30': 30,
    '6': 6,
    'log(210)': np.log(210),
    'log(48)': np.log(48),
    'e': np.e,
}

# Check which candidates match R₀_q, R₀_l, ratio, product, etc.
targets = {
    'R₀_q': R0_q_pop,
    'R₀_l': R0_l_pop,
    'L/Q': R0_ratio,
    'L×Q': R0_product,
    'L+Q': R0_sum,
    'L-Q': R0_diff,
    'Q²': R0_q_sq,
    'L²': R0_l_sq,
}

print(f"\n{'Target':>8} = {'Value':>12}  Best matches (<5%):")
for tname, tval in targets.items():
    matches = []
    for cname, cval in candidates_scan.items():
        if cval == 0:
            continue
        # Check tval/cval and tval*cval ratios for simple integer/fraction relationships
        ratio_tc = tval / cval
        for mult_name, mult in [('', 1), ('×2', 2), ('×3', 3), ('/2', 0.5), ('/3', 1/3),
                                  ('×π', np.pi), ('/π', 1/np.pi), ('×√', np.sqrt(ratio_tc))]:
            if mult_name == '×√':
                continue
            test = ratio_tc / mult if mult != 0 else 0
            if test > 0:
                # Check if close to integer or simple fraction
                for numer in range(1, 13):
                    for denom in range(1, 13):
                        frac = numer / denom
                        dev = abs(test - frac) / frac * 100
                        if dev < 3.0:
                            expr = f"{tname} ≈ ({numer}/{denom}){mult_name}×{cname}" if denom > 1 else f"{tname} ≈ {numer}{mult_name}×{cname}"
                            matches.append((dev, expr))
    matches.sort()
    if matches:
        for dev, expr in matches[:3]:
            print(f"  {tname:>8} = {tval:>12.6f}  {expr}  (dev={dev:.2f}%)")
    else:
        print(f"  {tname:>8} = {tval:>12.6f}  no clean match")

POPULATION R₀ NUMBER-THEORETIC SCAN
R₀_q = 5.81269217
R₀_l = 6.17597016
L/Q  = 1.06249737
L×Q  = 35.89901338
L+Q  = 11.98866233
L-Q  = 0.36327799
Q²   = 33.78739025
L²   = 38.14260741

  Target =        Value  Best matches (<5%):
      R₀_q =     5.812692  R₀_q ≈ (10/9)×π×p₃/p₂  (dev=0.09%)
      R₀_q =     5.812692  R₀_q ≈ (12/11)/3×d(210)  (dev=0.09%)
      R₀_q =     5.812692  R₀_q ≈ (2/11)×2×d(210)  (dev=0.09%)
      R₀_l =     6.175970  R₀_l ≈ (10/8)/2×π²  (dev=0.12%)
      R₀_l =     6.175970  R₀_l ≈ (5/4)/2×π²  (dev=0.12%)
      R₀_l =     6.175970  R₀_l ≈ (5/8)×π²  (dev=0.12%)
       L/Q =     1.062497  L/Q ≈ (3/11)/2×49/2π  (dev=0.09%)
       L/Q =     1.062497  L/Q ≈ (5/8)/π×log(210)  (dev=0.12%)
       L/Q =     1.062497  L/Q ≈ (3/7)/π×49/2π  (dev=0.13%)
       L×Q =    35.899013  L×Q ≈ (5/7)×π×d(210)  (dev=0.01%)
       L×Q =    35.899013  L×Q ≈ (12/4)×π×210^(1/4)  (dev=0.06%)
       L×Q =    35.899013  L×Q ≈ (6/2)×π×210^(1/4)  (dev=0.06%)
       L+Q =    11.988662  L+Q ≈ (

In [21]:
# ── Focused scan: R₀ algebraic structure ──
# Direct number-theoretic tests on best population values
R0q, R0l = pop_R0[('QUARK',3)], pop_R0[('LEPTON',3)]
print(f"R₀_q = {R0q:.8f}")
print(f"R₀_l = {R0l:.8f}")
print(f"R₀_l / R₀_q = {R0l/R0q:.8f}")

# Check: R₀ ≈ simple function of {2,3,5,7,π}?
tests = [
    ("7-1/p₄", 7 - 1/7),
    ("p₂·p₃/p₄·2π", 3*5/(7*2*np.pi)),
    ("2π-1/2", 2*np.pi - 0.5),
    ("p₄-1/p₃", 7-1/5),
    ("7·5/6", 7*5/6),
    ("log(210)·log(48)", np.log(210)*np.log(48)),
    ("√35", np.sqrt(35)),
    ("(p₄²-1)/2π", (49-1)/(2*np.pi)),
    ("48/2π", 48/(2*np.pi)),
    ("φ(P₄)/2π", 48/(2*np.pi)),
    ("(p₄²+1)/2π", 50/(2*np.pi)),
    ("p₄·5/6", 35/6),
    ("π·p₂-3/p₄", np.pi*3-3/7),
    ("2p₂·2p₃/(2p₄-1)", 6*10/13),
    ("√(P₄)/√7", np.sqrt(210/7)),
    ("√30", np.sqrt(30)),
    ("6+ρ", 6+1/np.sqrt(210)),
    ("7-ρ", 7-1/np.sqrt(210)),
    ("p₃+1/p₂", 5+1/3),
    ("37/p₄", 37/7),
    ("41/p₄", 41/7),
    ("p₃+p₂/p₄", 5+3/7),
    ("φ+1/2π", 49/(2*np.pi)),
    ("λ(35)/2", 6.0),
    ("2π·ρ", 2*np.pi/np.sqrt(210)),
    ("12/ρ", 12*np.sqrt(210)),
    ("P₄^(1/3)", 210**(1/3)),
    ("35/6", 35/6),
    ("36/p₄", 36/7),
    ("43/p₄", 43/7),
]

print(f"\n{'Expression':>20} {'Value':>12} {'dev_Q':>8} {'dev_L':>8}")
print(f"{'-'*52}")
for name, val in sorted(tests, key=lambda x: min(abs(x[1]-R0q)/R0q, abs(x[1]-R0l)/R0l)):
    dq = (val - R0q)/R0q * 100
    dl = (val - R0l)/R0l * 100
    if min(abs(dq), abs(dl)) < 5:
        print(f"  {name:>20} {val:>12.6f} {dq:>+8.2f}% {dl:>+8.2f}%")

# Check ratio L/Q against simple fractions
lr = R0l / R0q
print(f"\nR₀_l/R₀_q = {lr:.8f}")
for n in range(1, 20):
    for d in range(1, 20):
        if abs(n/d - lr) / lr < 0.01:
            print(f"  ≈ {n}/{d} = {n/d:.6f} (dev {(n/d-lr)/lr*100:+.3f}%)")

R₀_q = 5.81269217
R₀_l = 6.17597016
R₀_l / R₀_q = 1.06249737

          Expression        Value    dev_Q    dev_L
----------------------------------------------------
                 7·5/6     5.833333    +0.36%    -5.55%
                p₄·5/6     5.833333    +0.36%    -5.55%
                  35/6     5.833333    +0.36%    -5.55%
                2π-1/2     5.783185    -0.51%    -6.36%
                 43/p₄     6.142857    +5.68%    -0.54%
                 41/p₄     5.857143    +0.76%    -5.16%
                   6+ρ     6.069007    +4.41%    -1.73%
                   √35     5.916080    +1.78%    -4.21%
              P₄^(1/3)     5.943922    +2.26%    -3.76%
               λ(35)/2     6.000000    +3.22%    -2.85%

R₀_l/R₀_q = 1.06249737
  ≈ 15/14 = 1.071429 (dev +0.841%)
  ≈ 16/15 = 1.066667 (dev +0.392%)
  ≈ 17/16 = 1.062500 (dev +0.000%)
  ≈ 18/17 = 1.058824 (dev -0.346%)
  ≈ 19/18 = 1.055556 (dev -0.653%)


## NB74 Results: The Anatomy of CP Asymmetry

### Structural Findings

**1. Late-time equalization (R → 1)**: The perturbed solenoid's restoring force
−κR_k/p_k drives all covering residuals toward 0. Since both CP partners experience
the same restoring force, their RMS residuals equalize at late times → R(T → ∞) = 1.0
exactly. Verified: all per-window ratios are 1.0000 for T > 200.

**2. Window-0 concentration**: 100% of the CP asymmetry is concentrated in the
initial crossing regime (first ~200 time units, ~200 crossings). This is the
**only** phase where the two CP partners have different RMS residuals. After window 0,
both partners relax identically.

**3. Dilution model**: The cumulative R(T) from NB70-73 is explained by:

$$R_{\text{cum}}^2(T) = \frac{\sigma_1^2 + (n-1)\sigma_\infty^2}{\sigma_2^2 + (n-1)\sigma_\infty^2}$$

where σ₁, σ₂ are the window-0 partner RMS values and σ∞ is the common late-time RMS.
Verified: model matches observations to ~1% at T=5000.

### The Physical Picture

The mass predictions from NB70-73 work because:
- The initial-condition geometry of the solenoid creates **differential sensitivity**
  to the first ~200 crossings based on CRT bin assignment (a3, a7* → specific crossing numbers)
- Different CP partners sample different crossing numbers in the initial transient
- The perturbation couples levels non-uniformly → asymmetric residuals emerge
- At T=5000, the cumulative R₄ is the window-0 asymmetry diluted by ~24 symmetric windows
- The mass exponents from NB72-73 thus encode the **initial-condition spectral geometry**
  of the solenoid, not the late-time equilibrium

### Number-Theoretic Structure

| Quantity | Population Value | Best Match | Deviation |
|----------|-----------------|------------|-----------|
| R₀_q (quark, level 4) | 5.8127 | 35/6 = p₃p₄/P₂ | +0.36% |
| R₀_l (lepton, level 4) | 6.1760 | 43/7 | −0.54% |
| **R₀_l / R₀_q** | **1.06250** | **17/16 = (d(210)+1)/d(210)** | **0.000%** |

The ratio L/Q = 17/16 to 2.5 ppm is the most precise match. Whether this is a genuine
algebraic identity or a coincidence at current precision requires:
- Perturbation theory to derive R₀ analytically from the ODE
- Or: confirmation that the match persists with different perturbation parameters

**Scope boundary**: The individual R₀ values do not match clean algebraic expressions
in {2,3,5,7,π}. Closed-form derivation requires solving the initial-value perturbation
structure analytically — a task for future work.

In [23]:
# ── NB74 SCORECARD ──
R0_ratio_val = pop_R0[('LEPTON',3)] / pop_R0[('QUARK',3)]
sc = []
sc.append("#146 CP late-equalization: R(T→∞)=1.0 all channels — PASS")
sc.append("#147 Window-0 concentration: 100% CP asymmetry in first 200t — PASS")
sc.append(f"#148 Dilution model: Q={R_q_model:.4f} L={R_l_model:.4f} vs NB73 1.479/1.980 (~1%) — PASS")
sc.append(f"#149 R₀ ratio: {R0_ratio_val:.8f} vs 17/16={17/16:.8f} ({abs(R0_ratio_val-17/16)/(17/16)*100:.4f}%) — PROVISIONAL")
print("NB74 SCORECARD")
print("=" * 65)
for s in sc:
    print(f"  {s}")
print(f"\nNB74: 3 PASS + 1 PROVISIONAL (#146-#149)")
print(f"Running total: 142 identities, 74 notebooks, 0 free parameters")

NB74 SCORECARD
  #146 CP late-equalization: R(T→∞)=1.0 all channels — PASS
  #147 Window-0 concentration: 100% CP asymmetry in first 200t — PASS
  #148 Dilution model: Q=1.4632 L=1.9510 vs NB73 1.479/1.980 (~1%) — PASS
  #149 R₀ ratio: 1.06249737 vs 17/16=1.06250000 (0.0002%) — PROVISIONAL

NB74: 3 PASS + 1 PROVISIONAL (#146-#149)
Running total: 142 identities, 74 notebooks, 0 free parameters
